In [6]:
from sklearn.datasets import load_breast_cancer
X,y = load_breast_cancer(return_X_y=True)

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
X_train.shape

(455, 30)

In [9]:
import optuna
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

def Objective(trial):

    n_estimators = trial.suggest_int("n_estimators", 100,1000)
    max_depth = trial.suggest_int("max_depth",1,30)
    learning_rate = trial.suggest_float("learning_rate",0.1,0.3,log=True)
    subsample = trial.suggest_float("subsample",0.3,1.0)
    #Rowsampling
    colsample_bytree = trial.suggest_float("colsample_bytree",0.5,1.0)
    #Colsampling
    min_child_weight = trial.suggest_float("min_child_weight",1,10)
    gamma = trial.suggest_float("gamma", 0, 5)
    #Is it worth it to split?
    reg_alpha = trial.suggest_float("reg_alpha", 1e-8, 10, log=True)
    #L1 regularization
    reg_lambda = trial.suggest_float("reg_lambda", 1e-8, 10, log=True)
    #L2 regularization

    model = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_child_weight=min_child_weight,
        gamma=gamma,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda
    )

    return cross_val_score(model,X_train,y_train,cv=5,scoring='accuracy').mean()


In [10]:
study = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler())
#To create a study, maximize accuracy, and TPE sampler is the algorithm which would do this
study.optimize(Objective,n_trials=50)

[I 2026-09-16 15:19:40,000] A new study created in memory with name: no-name-8d44bf60-e692-4bad-9334-92818836d028
[I 2026-09-16 15:19:40,294] Trial 0 finished with value: 0.9538461538461538 and parameters: {'n_estimators': 112, 'max_depth': 3, 'learning_rate': 0.23761054700181075, 'subsample': 0.39953617674754194, 'colsample_bytree': 0.5544623736876592, 'min_child_weight': 3.438783669383498, 'gamma': 2.8358020524813696, 'reg_alpha': 0.6288196226796794, 'reg_lambda': 1.7846711204838547e-05}. Best is trial 0 with value: 0.9538461538461538.
[I 2026-09-16 15:19:40,945] Trial 1 finished with value: 0.9516483516483516 and parameters: {'n_estimators': 378, 'max_depth': 12, 'learning_rate': 0.13146024249273067, 'subsample': 0.9078358436104887, 'colsample_bytree': 0.5937732787336114, 'min_child_weight': 9.602312339491274, 'gamma': 3.493684884065945, 'reg_alpha': 0.5839498703244284, 'reg_lambda': 0.0004117575405715516}. Best is trial 0 with value: 0.9538461538461538.
[I 2026-09-16 15:19:41,401] 

In [11]:
study.best_trial.value

0.9736263736263737

In [12]:
study.best_trial.params

{'n_estimators': 255,
 'max_depth': 23,
 'learning_rate': 0.29727953308844574,
 'subsample': 0.8201458768985334,
 'colsample_bytree': 0.9365979266783271,
 'min_child_weight': 2.387094593986336,
 'gamma': 3.60613908909114,
 'reg_alpha': 0.0001430229707141589,
 'reg_lambda': 0.009508055356557096}

In [19]:
from sklearn.metrics import accuracy_score
best_model = XGBClassifier(**study.best_params,random_state=42)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
accuracy_score(y_test, y_pred)

0.9649122807017544

In [20]:
from optuna.visualization import plot_optimization_history,plot_parallel_coordinate,plot_slice,plot_contour,plot_param_importances

In [ ]:
plot_optimization_history(study).show()
#Here objective value is acc and X-axis is trial number

In [ ]:
plot_parallel_coordinate(study).show()
#Relates acc to parameters

In [23]:
plot_slice(study).show()

In [24]:
plot_contour(study).show()

In [25]:
plot_param_importances(study).show()